# 🌸 Ejercicio: Clasificación de Iris con MLflow

## 🎯 Objetivos del Ejercicio

En este ejercicio práctico aprenderás a:

1. **Trabajar con un problema de clasificación** multiclase
2. **Aplicar MLflow** a un modelo de árbol de decisión
3. **Registrar experimentos** de forma profesional
4. **Evaluar y comparar** diferentes configuraciones
5. **Crear visualizaciones** para clasificación

---

## 🌺 El Dataset Iris

El **Iris Dataset** es uno de los datasets más famosos en Machine Learning, introducido por Ronald Fisher en 1936.

### 📊 Características del Dataset

| Aspecto | Descripción |
|---------|-------------|
| **Muestras** | 150 flores (50 por clase) |
| **Características** | 4 medidas físicas (cm) |
| **Clases** | 3 especies de iris |
| **Tipo** | Clasificación multiclase |
| **Dificultad** | ⭐⭐ (Principiante) |

### 🌸 Las 3 Especies de Iris

1. **Iris Setosa** (Clase 0)
2. **Iris Versicolor** (Clase 1)
3. **Iris Virginica** (Clase 2)

### 📏 Las 4 Características

1. **Sepal Length** (Longitud del sépalo)
2. **Sepal Width** (Ancho del sépalo)
3. **Petal Length** (Longitud del pétalo)
4. **Petal Width** (Ancho del pétalo)

---

## 🎯 Tu Misión

Construir un **Árbol de Decisión** que pueda clasificar correctamente las especies de iris basándose en sus medidas físicas, y documentar todo el proceso con MLflow.

---

## 🚀 ¡Empecemos!

In [0]:
import mlflow
import warnings
warnings.filterwarnings('ignore')


mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks")

# ⚠️ IMPORTANTE: Cambia esto por tu email de Databricks
email = 'alvaroarquero1@gmail.com'  # Ejemplo: 'nombre.apellido@ejemplo.com'

# Validar que el email no esté vacío
if not email:
    print("⚠️  ADVERTENCIA: Debes configurar tu email antes de continuar")
    print("   Cambia la variable 'email' por tu email de Databricks")
else:
    # Configurar el experimento
    experiment_name = f"/Users/{email}/4-ejercicio-the-irish"
    mlflow.set_experiment(experiment_name)
    
    print("=" * 60)
    print("✅ MLflow configurado correctamente")
    print("=" * 60)
    print(f"📊 Experimento: {experiment_name}")
    print(f"🌸 Dataset: Iris (Clasificación)")
    print(f"🤖 Modelo: Decision Tree Classifier")
    print("=" * 60)

## ⚙️ Paso 1: Configuración de MLflow

Configuramos el experimento donde se registrarán todas las ejecuciones.

**🔴 IMPORTANTE**: Cambia el email vacío por tu email de Databricks.

## 📚 Paso 2: Importar Librerías

Importamos todas las herramientas necesarias para clasificación.

In [0]:
%pip install seaborn
#Meto esta línea ya que me daba error al ejecutar los imports de la siguiente celda (me daba un error y me he ayudado de la IA, ya que no conseguía identificar el error "ModuleNotFoundError: No module named 'seaborn'""")

In [0]:
# Librerías principales
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# MLflow para tracking
import mlflow
import mlflow.sklearn

# Scikit-learn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import datasets
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score,
    confusion_matrix,
    classification_report
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("Set2")

print("✅ Todas las librerías importadas correctamente")
print("   - MLflow: Listo para tracking")
print("   - Scikit-learn: Modelos y métricas")
print("   - Matplotlib & Seaborn: Visualizaciones")

## 🌺 Paso 3: Cargar y Explorar el Dataset Iris

Cargaremos el famoso dataset de flores Iris y exploraremos sus características.

In [0]:
from sklearn import datasets
import pandas as pd
import matplotlib.pyplot as plt


In [0]:
# TODO: Carga el dataset Iris y explora su estructura.
# Pistas:
# - Usa datasets.load_iris()
# - Crea un DataFrame con dataset.data y dataset.feature_names
# - Añade la columna target/especie
# - Muestra shape, clases, primeras filas y estadísticas descriptivas

dataset = datasets.load_iris()
df = pd.DataFrame(dataset.data, columns=dataset.feature_names)
df['especie'] = dataset.target
print('Shape:', df.shape)
print('Clases', dataset.target_names)
print(df.head())
print(df.describe())


### 📊 Visualización Exploratoria del Dataset

Visualicemos las relaciones entre las características para entender mejor los datos.

In [0]:
import seaborn as sns

In [0]:
#### TODO: Crea visualizaciones exploratorias del dataset Iris.
# Ideas:
# - Relación entre longitud/ancho del pétalo por especie
# - Distribución de características por especie
# - Boxplots por especie
# - Matriz de correlación
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
#Primer gráfico para ver la relacion entre longitud/ancho del petalo de cada especie
sns.scatterplot(data = df, x='sepal length (cm)', y='sepal width (cm)', hue='especie', ax=axes[0,0])
#Segundo gráfico
sns.histplot(data=df, x='sepal length (cm)', hue='especie', ax=axes[0, 1])
#Boxplot por especie
sns.boxplot(data=df, x='especie', y='petal length (cm)', ax=axes[1, 0])
#Matriz correlacion
sns.heatmap(df[dataset.feature_names].corr(), annot=True, ax=axes[1, 1])
plt.show()

# TODO: Escribe tus observaciones clave sobre separabilidad y correlaciones.
#Las distintas especies se pueden distinguir bien empleando las características de los pétalos. Las características del sepalo presentan mucho más solapamiento entre las 3 especies


## 🔀 Paso 4: Preparar los Datos

Dividimos el dataset en conjuntos de entrenamiento y prueba.

In [0]:
# TODO: Separa características (X) y etiquetas (y).
X = df[dataset.feature_names]
y = df['especie']

# TODO: Divide el dataset en entrenamiento y prueba.
# Pistas:
# - Usa train_test_split
# - Reserva una parte para test
# - Usa random_state para reproducibilidad
# - Considera stratify=y para mantener la proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# TODO: Comprueba tamaños y distribución de clases en cada conjunto.
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)
print("Distribución en entrenamiento:")
print(y_train.value_counts())
print("Distribución en prueba:")
print(y_test.value_counts())


## 🌳 Paso 5: Entrenar Árbol de Decisión con MLflow

Ahora entrenaremos un modelo de **Decision Tree** y registraremos todo con MLflow.

### 🔑 Hiperparámetros del Árbol de Decisión

- **max_depth**: Profundidad máxima del árbol (evita overfitting)
- **max_features**: Número máximo de características a considerar por split
- Valores más altos = modelo más complejo = mayor riesgo de overfitting

In [0]:
# ========================================
# TODO: ENTRENAMIENTO CON MLFLOW
# ========================================

# TODO: Activa autologging de MLflow.
mlflow.sklearn.autolog()

# TODO: Inicia un run con un nombre descriptivo.
with mlflow.start_run(run_name="Decision Tree - Iris Classifier") as run:
#     TODO: Define hiperparámetros del árbol.
     max_depth = 3
     max_features = None
     random_state = 42

#     TODO: Crea y entrena DecisionTreeClassifier.
     dt = DecisionTreeClassifier(
        max_depth=max_depth,
        max_features=max_features,
        random_state=random_state
     )

     dt.fit(X_train, y_train)

#     TODO: Genera predicciones para train y test.
     y_pred_train = dt.predict(X_train)
     y_pred_test = dt.predict(X_test)

#     TODO: Calcula métricas de clasificación.
     accuracy_test = accuracy_score(y_test, y_pred_test)
     precision_test = precision_score(y_test, y_pred_test, average='weighted')
     recall_test = recall_score(y_test, y_pred_test, average='weighted')
     f1_test = f1_score(y_test, y_pred_test, average='weighted')
#
#     TODO: Evalúa con cross-validation.
     cv_scores = cross_val_score(dt, X, y, cv=5)
     print("Cross-validation:", cv_scores)
     print("Media:", cv_scores.mean())
#
#     TODO: Registra métricas/parámetros relevantes en MLflow si no quedan registrados automáticamente.
     mlflow.log_metric("accuracy_test", accuracy_test)
     mlflow.log_metric("precision_test", precision_test)
     mlflow.log_metric("recall_test", recall_test)
     mlflow.log_metric("f1_test", f1_test)
     mlflow.log_metric("cv_mean", cv_scores.mean())
#
#     TODO: Visualiza matriz de confusión y, opcionalmente, el árbol.
     cm = confusion_matrix(y_test, y_pred_test)

     sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
     plt.xlabel('Predicción')
     plt.ylabel('Real')
     plt.title('Matriz de confusión')
     plt.show()
#
#     TODO: Escribe una breve interpretación de los resultados.
#Los resultados del cross validation son similares a los obtenidos en el conjunto de pruebas, indicando asi un rendimiento estable

## 🎯 Reflexión del Ejercicio

Completa esta sección cuando termines el notebook.

### ✅ Comprueba que has trabajado

1. [ ] Exploración del dataset Iris
2. [ ] Separación train/test
3. [ ] Entrenamiento de un Árbol de Decisión
4. [ ] Evaluación con métricas de clasificación
5. [ ] Registro del experimento en MLflow
6. [ ] Interpretación de resultados

---

## 📚 Conceptos Clave - Clasificación

### 🎯 Métricas de Clasificación

| Métrica | Descripción | Cuándo Usarla |
|---------|-------------|---------------|
| **Accuracy** | % de predicciones correctas | Clases balanceadas |
| **Precision** | % de positivos correctos | Importante evitar falsos positivos |
| **Recall** | % de positivos encontrados | Importante encontrar todos los positivos |
| **F1-Score** | Media armónica de P y R | Balance entre precisión y recall |

### 🌳 Árbol de Decisión

**TODO: Explica con tus palabras:**
- ¿Qué ventajas tiene un árbol de decisión?: No necesita muchos datos, y permite visualizar las decisiones que toma el modelo
- ¿Qué riesgos tiene respecto a overfitting?: Si el arbol crece demasiado, puede aprender demasiados datos innecesariamente
- ¿Cómo interpretarías una matriz de confusión multiclase?: los valores de la diagonal son las prediccions correctas

---

## 🚀 Desafíos Adicionales

### 🎯 Desafío 1: Optimiza los Hiperparámetros

Prueba diferentes configuraciones y encuentra la mejor:

```python
# Experimenta con:
max_depth = [3, 5, 10, None]
max_features = [2, 3, 4, 'sqrt']
min_samples_split = [2, 5, 10]
```

**TODO:** ¿Qué combinación da el mejor balance entre accuracy y overfitting?

### 🎯 Desafío 2: Implementa Grid Search

Automatiza la búsqueda del mejor modelo con `GridSearchCV`.

### 🎯 Desafío 3: Compara con Otros Modelos

Entrena y compara Random Forest, SVM, KNN o Logistic Regression.

### 🎯 Desafío 4: Análisis de Errores

**TODO:** ¿Qué flores se confunden más y por qué?

### 🎯 Desafío 5: Reducción de Dimensionalidad

Usa PCA para reducir a 2 dimensiones y visualizar límites de decisión.


In [0]:
# 🎨 DESAFÍOS OPCIONALES
# Usa este espacio para completar los desafíos del ejercicio.

# ========================================
# TODO: DESAFÍO 1 - Optimización manual
# ========================================
configuraciones = [3, 5, 10, None]
for config in configuraciones:
    modelo = DecisionTreeClassifier(
        max_depth=config,
        random_state=42
    )
#     # Entrena, evalúa y registra cada configuración en MLflow
#     pass
modelo.fit(X_train, y_train)
accuracy_train = modelo.score(X_train, y_train)
accuracy_test = modelo.score(X_test, y_test)

# ========================================
# TODO: DESAFÍO 2 - Grid Search
# ========================================
from sklearn.model_selection import GridSearchCV
param_grid = {
    'max_depth': [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
}
grid_search = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5
)
grid_search.fit(X_train, y_train)

# ========================================
# TODO: DESAFÍO 3 - Comparación de modelos
# ========================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
modelos = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "KNN": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=200)
}
for nombre, modelo in modelos.items():
#     # Entrena y compara métricas
#     pass
    modelo.fit(X_train, y_train)
    accuracy = modelo.score(X_test, y_test)
    print(f"{nombre}: {accuracy}")

# ========================================
# TODO: DESAFÍO 4 - Análisis de errores
# ========================================
incorrect_indices = y_test != y_pred_test
# TODO: inspecciona las muestras mal clasificadas.
errores = X_test[incorrect_indices].copy()

errores['Real'] = y_test[incorrect_indices]
errores['Prediccion'] = y_pred_test[incorrect_indices]

print(errores)

# ========================================
# TODO: DESAFÍO 5 - PCA y visualización 2D
# ========================================
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=y,
)